In [11]:
# Додавання бібліотек
import math
import pprint
import random
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd

# Можливі значення змінних
tss_range = (17, 27)
ta_range = (6, 17)
ph_range = (2.8, 4.0)
grape_kinds = ["blue", "green"]

# Кількість екземплярів
samples = 14
# Генеровані стовпці
generated_data = {
    # Вид винограду
    "kind": [random.choice(grape_kinds) for _ in range(samples)],
    # Точна кислотність
    "ph": [round(random.uniform(*ph_range), 1) for _ in range(samples)],
    # Відносна кислотність
    "ta": [random.randint(*ta_range) for _ in range(samples)],
    # Загальна кількість розчинних твердих речовин
    "tss": [random.randint(*tss_range) for _ in range(samples)],
}
kind_labels = {value: number for number, value in enumerate(grape_kinds)}
print(kind_labels)


{'blue': 0, 'green': 1}


In [12]:
df = pd.DataFrame(data=generated_data)
print(df)

     kind   ph  ta  tss
0    blue  3.5  15   22
1    blue  3.2  15   23
2   green  3.5  15   20
3    blue  3.4  14   19
4   green  3.1  12   25
5   green  2.8   6   21
6   green  3.9   8   26
7    blue  3.7   9   20
8   green  3.0   8   26
9   green  3.3   9   20
10   blue  3.3   9   21
11  green  2.8  10   18
12   blue  3.3   8   24
13  green  3.5   8   19


In [13]:
for sample_number, sample in df.iterrows():
    df.at[sample_number, "kind"] = kind_labels[df["kind"][sample_number]]

feature_sums = dict()
for feature_name, feature_samples in df.items():
    if feature_name == "kind":
        continue

    df[feature_name] = df[feature_name].astype(float)
    samples_sum = math.sqrt(sum(sample**2 for sample in feature_samples))
    for sample_number, sample_value in enumerate(feature_samples):
        normalized_value = sample_value / samples_sum
        df.at[sample_number, feature_name] = round(normalized_value, 2)

    feature_sums[feature_name] = samples_sum
print(df)

   kind    ph    ta   tss
0     0  0.28  0.37  0.27
1     0  0.26  0.37  0.28
2     1  0.28  0.37  0.24
3     0  0.27  0.34  0.23
4     1  0.25  0.30  0.31
5     1  0.23  0.15  0.26
6     1  0.31  0.20  0.32
7     0  0.30  0.22  0.24
8     1  0.24  0.20  0.32
9     1  0.27  0.22  0.24
10    0  0.27  0.22  0.26
11    1  0.23  0.25  0.22
12    0  0.27  0.20  0.29
13    1  0.28  0.20  0.23


In [14]:
class_centers = dict()
for class_number in kind_labels.values():
    center = {
        "ph": float(round(df[df["kind"] == class_number]["ph"].mean(), 2)),
        "ta": float(round(df[df["kind"] == class_number]["ta"].mean(), 2)),
        "tss": float(round(df[df["kind"] == class_number]["tss"].mean(), 2)),
    }
    class_centers[class_number] = center
pprint.pprint(class_centers)

{0: {'ph': 0.28, 'ta': 0.29, 'tss': 0.26},
 1: {'ph': 0.26, 'ta': 0.24, 'tss': 0.27}}


In [15]:
new_sample = {
    "ph": round(random.uniform(*ph_range), 1),
    "ta": random.randint(*ta_range),
    "tss": random.randint(*tss_range),
}
print(new_sample)

{'ph': 3.0, 'ta': 14, 'tss': 23}


In [16]:
for feature, value in new_sample.items():
    feature_sum = feature_sums[feature]
    new_sample[feature] = round(value / feature_sum, 2)
print(new_sample)

{'ph': 0.24, 'ta': 0.34, 'tss': 0.28}


In [17]:
distances = dict()
for class_number in class_centers.keys():
    center_values = np.array(list(class_centers[class_number].values()))
    new_sample_values = np.array(list(new_sample.values()))
    distance_to_class = math.sqrt(sum((new_sample_values - center_values) ** 2))
    distances[distance_to_class] = class_number
print(distances)

{0.06708203932499375: 0, 0.10246950765959602: 1}


In [18]:
sorted_distances = sorted(list(distances.keys()))
print(sorted_distances)

[0.06708203932499375, 0.10246950765959602]


In [19]:
predicted_class = 0
if len(sorted_distances) == 1 or sorted_distances[0] != sorted_distances[1]:
    predicted_class = distances[sorted_distances[0]]
elif sorted_distances[0] == sorted_distances[1]:
    number_of_samples_one = len(df[df["kind"] == distances[sorted_distances[0]]])
    number_of_samples_two = len(df[df["kind"] == distances[sorted_distances[1]]])
    predicted_class = (
        distances[sorted_distances[0]]
        if number_of_samples_one >= number_of_samples_two
        else distances[sorted_distances[1]]
    )
print(predicted_class)

0


In [ ]:
df.loc[-1] = [predicted_class] + list(new_sample.values())
df.index = df.index + 1
df = df.sort_index()
df["kind"] = df["kind"].astype(int)
print(df)

    kind    ph    ta   tss
0      0  0.24  0.34  0.28
1      0  0.28  0.37  0.27
2      0  0.26  0.37  0.28
3      1  0.28  0.37  0.24
4      0  0.27  0.34  0.23
5      1  0.25  0.30  0.31
6      1  0.23  0.15  0.26
7      1  0.31  0.20  0.32
8      0  0.30  0.22  0.24
9      1  0.24  0.20  0.32
10     1  0.27  0.22  0.24
11     0  0.27  0.22  0.26
12     1  0.23  0.25  0.22
13     0  0.27  0.20  0.29
14     1  0.28  0.20  0.23


: 